# Word2Vec
## Google's model
## Own model with raw data processing
## Own model with raw data using Simple Processing method

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from tqdm import tqdm
import gensim
from gensim.utils import simple_preprocess
from sklearn.model_selection import train_test_split
nltk.download("stopwords")
lemmatize = WordNetLemmatizer()
np.set_printoptions(edgeitems=30,linewidth=100000,formatter=dict(float=lambda x: "%.3g" % x))


[nltk_data] Downloading package stopwords to /home/san/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("/home/san/Obsidian/Data Science/NLP/assets/SMS_Spam_collection.txt", sep='\t',names=  ['label','message'])
df.head(2), df.shape

(  label                                            message
 0   ham  Go until jurong point, crazy.. Available only ...
 1   ham                      Ok lar... Joking wif u oni...,
 (5572, 2))

# Downloading google model

In [3]:
import gensim.downloader as api
google_wv = api.load('word2vec-google-news-300')

In [4]:
google_wv['king'], google_wv['king'].shape

(array([0.126, 0.0298, 0.00861, 0.14, -0.0256, -0.0361, 0.112, -0.198, 0.0513, 0.363, -0.242, -0.303, -0.178, -0.0249, -0.168, -0.17, 0.0347, 0.00522, 0.0464, 0.129, 0.137, 0.113, 0.0596, 0.137, 0.101, -0.177, -0.252, 0.0598, 0.342, -0.0311, 0.104, 0.0618, 0.125, 0.4, -0.322, 0.084, 0.0391, 0.00586, 0.0703, 0.173, 0.139, -0.231, 0.283, 0.143, 0.342, -0.0239, -0.11, 0.0332, -0.0547, 0.0153, -0.162, 0.158, -0.26, 0.0201, -0.163, 0.00136, -0.145, -0.0569, 0.043, -0.0247, 0.186, 0.447, 0.00958, 0.132, 0.0986, -0.186, -0.1, -0.134, -0.125, 0.283, 0.123, 0.0532, -0.178, 0.0859, -0.0219, 0.0205, -0.14, 0.0251, 0.139, -0.105, 0.139, 0.0889, -0.0752, -0.0214, 0.173, 0.0464, -0.266, 0.00891, 0.149, 0.0378, 0.238, -0.125, -0.218, -0.182, 0.0298, 0.0571, -0.0289, 0.0125, 0.0967, -0.231, 0.0581, 0.0669, 0.0708, -0.309, -0.215, 0.146, -0.428, -0.0094, 0.154, -0.0767, 0.289, 0.277, -0.000486, -0.137, 0.324, -0.246, -0.00304, -0.212, 0.125, 0.27, 0.204, 0.0825, -0.201, -0.16, -0.0378, -0.12, 0.115, -0

In [56]:
google_wv.similar_by_word('Santhosh')

[('Biju', 0.7292504906654358),
 ('Suresh', 0.7274524569511414),
 ('Sunil', 0.7210167646408081),
 ('Manoj', 0.7154952883720398),
 ('Babu', 0.7129248380661011),
 ('Rajesh', 0.7106234431266785),
 ('Mahesh', 0.70893394947052),
 ('Santosh', 0.704844057559967),
 ('Kishore', 0.7047179341316223),
 ('Prashanth', 0.702256441116333)]

In [5]:
df.head(2)

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...


In [6]:
df['message'] = df['message'].astype('str').str.lower()
df['label'] = df['label'].astype('str').str.lower()

df.head(2)

,label,message
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...


In [7]:
df['message'] = df.message.apply(lambda x: re.sub('[^a-z A-Z]',' ',x).strip())
df.head(2)

,label,message
0,ham,go until jurong point crazy available only ...
1,ham,ok lar joking wif u oni


In [8]:
stop_words = set(stopwords.words('english'))

df.message = df.message.apply(lambda x:' '.join(lemmatize.lemmatize(word) for word in x.split() if word not in stop_words))
df.head(2)

,label,message
0,ham,go jurong point crazy available bugis n great ...
1,ham,ok lar joking wif u oni


In [9]:
df['tokens'] = df['message'].apply(lambda x: x.split())
df.label = df.label.replace({"ham":1, "spam":0})
df.head()

/tmp/ipykernel_315913/670803129.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.label = df.label.replace({"ham":1, "spam":0})


,label,message,tokens
0,1,go jurong point crazy available bugis n great ...,"[go, jurong, point, crazy, available, bugis, n..."
1,1,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
2,0,free entry wkly comp win fa cup final tkts st ...,"[free, entry, wkly, comp, win, fa, cup, final,..."
3,1,u dun say early hor u c already say,"[u, dun, say, early, hor, u, c, already, say]"
4,1,nah think go usf life around though,"[nah, think, go, usf, life, around, though]"


In [10]:
X_train_g, X_test_g,y_train_g, y_test_g= train_test_split(df['tokens'],df['label'],test_size=0.2)

In [11]:
X_train_g[:2], X_test_g[:2]

(2898    [collecting, ur, laptop, going, configure, da,...
 775                                   [sleeping, surfing]
 Name: tokens, dtype: object,
 2197                  [much, textin, bout]
 4598    [full, heat, pa, applyed, oil, pa]
 Name: tokens, dtype: object)

In [12]:
y_train_g[:2], y_test_g[:2]

(2898    1
 775     1
 Name: label, dtype: int64,
 2197    1
 4598    1
 Name: label, dtype: int64)

In [13]:
# * NOTE: creating Avg word 2 vec
def avg_w2v(doc, model):
    vectors = [model[word] for word in doc if word in model]
    if not vectors:
        return np.zeros(model.vector_size)
    # print(f"raw_vector {vectors}") # ! first flattens all arrays into one long list of numbers (if you have 10 words, that's 10 × 100 = 1000 numbers) and then computes the single mean of all those numbers. 
    # print(f"raw_mean {np.mean(vectors)}")
    # print(f"axis=0_mean {np.mean(vectors,axis=0)}")
    return np.mean(vectors,axis=0)


 # !Example with small numbers
 # !
 # !Suppose each word vector is 3‑dimensional (just for illustration):
 # !
     # !Word1: [1, 2, 3]
 # !
     # !Word2: [4, 5, 6]
 # !
     # !Word3: [7, 8, 9]
 # !
 # !np.mean(vectors) = average of all 9 numbers = (1+2+3+4+5+6+7+8+9)/9 = 5.0 (a scalar).
 # !
 # !np.mean(vectors, axis=0) = average column‑wise:
 # !
     # !Dim0: (1+4+7)/3 = 4
 # !
     # !Dim1: (2+5+8)/3 = 5
 # !
     # !Dim2: (3+6+9)/3 = 6
     # !Result: [4, 5, 6] (a 3‑d vector).

In [14]:
X_train_vectors = np.array([avg_w2v(tokens, google_wv) for tokens in X_train_g])
X_test_vectors  = np.array([avg_w2v(tokens, google_wv) for tokens in X_test_g])

print("Train shape:", X_train_vectors.shape)
print("Test shape :", X_test_vectors.shape)

Train shape: (4457, 300)
Test shape : (1115, 300)


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, recall_score,f1_score

In [16]:
RF_Classifier = RandomForestClassifier()

RF_Classifier.fit(X_train_vectors,y_train_g)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [17]:
y_pred = RF_Classifier.predict(X_test_vectors)
y_pred_train = RF_Classifier.predict(X_train_vectors)
print(classification_report(y_pred,y_test_g))

{"Acc_test": accuracy_score(y_test_g,y_pred),"Acc_train":accuracy_score(y_pred_train,y_train_g),"F1_test":f1_score(y_pred,y_test_g,average=None)}

              precision    recall  f1-score   support

           0       0.77      1.00      0.87       118
           1       1.00      0.96      0.98       997

    accuracy                           0.97      1115
   macro avg       0.88      0.98      0.92      1115
weighted avg       0.98      0.97      0.97      1115



{'Acc_test': 0.967713004484305,
 'Acc_train': 0.9997756338344178,
 'F1_test': array([0.868, 0.982])}

In [18]:
from sklearn.model_selection import  GridSearchCV, cross_val_score, RandomizedSearchCV, StratifiedKFold
param_grid = {
    "n_estimators" : [50, 100,300],
    "criterion":["gini", "entropy"],
    "min_samples_split": [2,5,10],
    "min_samples_leaf":[1,2, 4],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
    "ccp_alpha":[0.001, 0.005],
    'max_depth': [10, 20, 30, None],

}

In [19]:
rf = RandomForestClassifier(random_state=42)
random_search = RandomizedSearchCV(rf, param_grid, n_iter=30, cv=5, scoring='f1_macro', n_jobs=-1)
random_search.fit(X_train_vectors, y_train_g)
print(random_search.best_params_)

{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.3, 'max_depth': None, 'criterion': 'entropy', 'ccp_alpha': 0.001}


In [20]:
RF_Classifier_tuned = RandomForestClassifier(n_estimators=300,min_samples_split=5,min_samples_leaf=4, max_features=0.3,max_depth=None, criterion="entropy",ccp_alpha=0.001,)

RF_Classifier_tuned.fit(X_train_vectors,y_train_g)

y_pred = RF_Classifier_tuned.predict(X_test_vectors)
y_pred_train = RF_Classifier_tuned.predict(X_train_vectors)
print(classification_report(y_pred,y_test_g))

{"Acc_test": accuracy_score(y_test_g,y_pred),"Acc_train":accuracy_score(y_pred_train,y_train_g),"F1_test":f1_score(y_pred,y_test_g,average=None)}

              precision    recall  f1-score   support

           0       0.79      0.98      0.87       123
           1       1.00      0.97      0.98       992

    accuracy                           0.97      1115
   macro avg       0.89      0.98      0.93      1115
weighted avg       0.97      0.97      0.97      1115



{'Acc_test': 0.968609865470852,
 'Acc_train': 0.9968588736818488,
 'F1_test': array([0.874, 0.982])}

## Own model with raw data processing

In [21]:
lemmatize = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
corpus = []
for i in range(0,len(df['message'])):

    sent = re.sub('[^a-z A-Z]',' ',df['message'].iat[i])
    sent_lower = sent.lower().split()
    after_lem = [lemmatize.lemmatize(i) for i in sent_lower if i not in stop_words]
    corpus.append(after_lem)
corpus[:3], len(corpus)

([['go',
   'jurong',
   'point',
   'crazy',
   'available',
   'bugis',
   'n',
   'great',
   'world',
   'la',
   'e',
   'buffet',
   'cine',
   'got',
   'amore',
   'wat'],
  ['ok', 'lar', 'joking', 'wif', 'u', 'oni'],
  ['free',
   'entry',
   'wkly',
   'comp',
   'win',
   'fa',
   'cup',
   'final',
   'tkts',
   'st',
   'may',
   'text',
   'fa',
   'receive',
   'entry',
   'question',
   'std',
   'txt',
   'rate',
   'c',
   'apply']],
 5572)

In [34]:
X_train, X_test,y_train, y_test= train_test_split(corpus,df['label'],test_size=0.2)
X_train[:2]

[['poyyarikatur',
  'kolathupalayam',
  'unjalur',
  'post',
  'erode',
  'dis',
  'lt',
  'gt'],
 ['mm', 'food', 'da']]

In [26]:
raw_model = gensim.models.Word2Vec(X_train,epochs=200,window=10)

In [27]:
{"count":raw_model.corpus_count,"vector_size":raw_model.vector_size}

{'count': 4457, 'vector_size': 100}

In [28]:
raw_model.wv.similar_by_word('rate'),raw_model.wv['rate'].shape

([('national', 0.6941889524459839),
  ('croydon', 0.47843053936958313),
  ('standard', 0.46402767300605774),
  ('normal', 0.4470350444316864),
  ('wb', 0.4402766823768616),
  ('operator', 0.4284312427043915),
  ('polyphonic', 0.41955602169036865),
  ('box', 0.4156038761138916),
  ('account', 0.4048045575618744),
  ('vodafone', 0.39657485485076904)],
 (100,))

In [29]:
# ! check all the vocabulary
print(raw_model.wv.index_to_key)

['u', 'call', 'ur', 'get', 'gt', 'lt', 'go', 'ok', 'free', 'day', 'know', 'come', 'good', 'time', 'got', 'like', 'love', 'text', 'want', 'send', 'need', 'today', 'one', 'r', 'c', 'n', 'stop', 'p', 'txt', 'lor', 'see', 'home', 'sorry', 'going', 'tell', 'still', 'take', 'back', 'k', 'mobile', 'think', 'da', 'reply', 'week', 'pls', 'later', 'hi', 'dont', 'phone', 'please', 'co', 'dear', 'new', 'night', 'make', 'well', 'much', 'min', 'msg', 'say', 'hope', 'thing', 'work', 'happy', 'oh', 'message', 'claim', 'number', 'great', 'yes', 'b', 'wat', 'friend', 'hey', 'give', 'www', 'let', 'way', 'tomorrow', 'amp', 'ask', 'yeah', 'prize', 'e', 'said', 'win', 'babe', 'really', 'meet', 'im', 'already', 'life', 'find', 'cash', 'right', 'tone', 'year', 'anything', 'uk', 'morning', 'pick', 'care', 'feel', 'last', 'would', 'urgent', 'every', 'cant', 'miss', 'thanks', 'lol', 'com', 'nokia', 'sure', 'contact', 'something', 'x', 'first', 'service', 'w', 'also', 'wait', 'box', 'even', 'buy', 'sm', 'help', '

In [31]:
for i in X_train:
    print(i)
    print(raw_model.wv[i[0]])
    break

['really', 'still', 'tonight', 'babe']
[-2.22 1.03 -2.02 1.83 -0.666 2.21 3.47 -1.17 0.477 -1.21 -3.31 -2.04 -0.407 0.788 -0.513 -1.71 -2.66 -1.37 -0.317 -0.883 0.928 5.32 0.8 1.73 -3.83 4.42 3.77 -1.67 -2.63 1.2 0.305 -2.77 0.123 1.7 2.07 -1.1 -1.67 2.97 0.566 0.983 -0.0245 -2.91 -3.33 2.25 -1.71 2.57 -0.735 1.24 -0.729 -0.0933 -2.59 0.727 1.22 0.08 -1.55 0.841 1.98 0.184 -0.736 0.734 1.45 -1.51 -3.42 -0.989 1.48 0.252 -0.0646 1.43 -1.57 -0.496 2.89 4 -0.733 2.62 -2.28 -1.19 1.06 0.115 0.709 0.396 -1.96 -2.93 2.31 -3 -0.365 -0.397 0.266 -1.14 -1.65 -1.71 -0.954 -0.585 -0.214 3.54 -0.645 -4.5 1.26 0.058 1.66 2.52]


In [45]:
def avg_w2v(tokens, model):
    """Return average vector for a list of tokens (ignores OOV words)."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Vectorize training set
X_train_vec_own = np.array([avg_w2v(doc, raw_model) for doc in X_train])

# Vectorize test set
X_test_vec_own = np.array([avg_w2v(doc, raw_model) for doc in X_test])

print("Train shape:", X_train_vec_own.shape)   # (n_train, 100)
print("Test shape :", X_test_vec_own.shape)    # (n_test,  100)

Train shape: (4457, 100)
Test shape : (1115, 100)


In [46]:
for i in X_train:
    print(i)
    print(avg_w2v(i,raw_model))
    break

['poyyarikatur', 'kolathupalayam', 'unjalur', 'post', 'erode', 'dis', 'lt', 'gt']
[-2.25 1.59 -0.607 -0.505 -1.32 0.949 -0.956 -1.05 -1.17 0.278 -0.159 1.75 -0.0325 -0.358 -1.85 -0.121 1.25 1.56 -1.19 -2.03 -0.529 0.16 0.524 1.8 -0.0673 -0.462 -0.946 -0.949 -0.616 0.961 1.09 -0.998 0.303 -0.983 1.07 -0.149 1.08 0.446 1.28 -0.943 2.12 0.694 0.19 -0.608 1.83 -0.283 -0.437 1.73 0.991 0.516 0.192 0.837 0.045 0.84 -0.817 -0.754 -0.531 -1.88 -0.166 -0.677 1.4 -0.601 0.357 -0.837 -0.498 0.155 1.27 0.0628 0.47 -0.736 -0.127 1.14 0.16 1.46 -0.187 0.185 -1.13 0.84 0.272 0.552 0.37 -1.22 0.352 -0.712 0.417 -1.11 -2.17 -0.21 -0.38 -0.627 -0.0694 1.02 0.113 -0.973 1.26 -1.41 0.919 -0.463 0.231 0.447]


In [47]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, recall_score,f1_score

In [49]:
RF_Classifier = RandomForestClassifier()

RF_Classifier.fit(X_train_vec_own,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [50]:
y_pred = RF_Classifier.predict(X_test_vec_own)
y_pred_train = RF_Classifier.predict(X_train_vec_own)

In [51]:
{"Acc_test": accuracy_score(y_test,y_pred),"Acc_train":accuracy_score(y_pred_train,y_train),"F1_test":f1_score(y_pred,y_test,average=None)}

{'Acc_test': 0.9856502242152466,
 'Acc_train': 0.9997756338344178,
 'F1_test': array([0.95, 0.992])}

In [64]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.97      0.91      0.94       167
           1       0.98      1.00      0.99       948

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.97      1115
weighted avg       0.98      0.98      0.98      1115



In [53]:
from sklearn.model_selection import  GridSearchCV, cross_val_score, RandomizedSearchCV, StratifiedKFold

In [54]:
param_grid = {
    "n_estimators" : [50, 100,300],
    "criterion":["gini", "entropy"],
    "min_samples_split": [2,5,10],
    "min_samples_leaf":[1,2, 4],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
    "ccp_alpha":[0.001, 0.005],
    'max_depth': [10, 20, 30, None],

}

In [55]:
rf = RandomForestClassifier(random_state=42)
random_search = RandomizedSearchCV(rf, param_grid, n_iter=30, cv=5, scoring='f1_macro', n_jobs=-1)
random_search.fit(X_train_vec_own, y_train)
print(random_search.best_params_)

{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30, 'criterion': 'entropy', 'ccp_alpha': 0.001}


In [56]:
RF_Classifier_tuned = RandomForestClassifier(n_estimators=100,min_samples_split=2,min_samples_leaf=1, max_features='sqrt',max_depth=30, criterion="entropy",ccp_alpha=0.001,)

RF_Classifier_tuned.fit(X_train_vec_own,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [57]:
y_pred = RF_Classifier_tuned.predict(X_test_vec_own)
y_pred_train = RF_Classifier_tuned.predict(X_train_vec_own)

In [58]:
{"Acc_test": accuracy_score(y_test,y_pred),"Acc_train":accuracy_score(y_pred_train,y_train),"F1_test":f1_score(y_pred,y_test,average=None)}

{'Acc_test': 0.9829596412556054,
 'Acc_train': 0.9984294368409243,
 'F1_test': array([0.941, 0.99])}

In [62]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.97      0.91      0.94       167
           1       0.98      1.00      0.99       948

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.97      1115
weighted avg       0.98      0.98      0.98      1115



In [60]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# y_train = y_train['spam'].astype(int).values  

accuracy_scores = cross_val_score(RF_Classifier_tuned, X_train_vec_own, y_train, cv=cv, scoring='accuracy')
f1_macro_scores = cross_val_score(RF_Classifier_tuned, X_train_vec_own, y_train, cv=cv, scoring='f1_macro')
f1_spam_scores = cross_val_score(RF_Classifier_tuned, X_train_vec_own, y_train, cv=cv, scoring='f1') 

print(f"Accuracy: {accuracy_scores.mean():.4f} ± {accuracy_scores.std():.4f}")
print(f"F1‑macro: {f1_macro_scores.mean():.4f} ± {f1_macro_scores.std():.4f}")
print(f"F1‑spam:  {f1_spam_scores.mean():.4f} ± {f1_spam_scores.std():.4f}")

Accuracy: 0.9773 ± 0.0046
F1‑macro: 0.9497 ± 0.0092
F1‑spam:  0.9877 ± 0.0025


# Own model with raw data using Simple Processing method

In [51]:


from nltk import sent_tokenize
from gensim.utils import simple_preprocess

